### EDA

In [26]:
import pandas as pd
import numpy as np

# list of  weather underground files (add more  needed)
files = [
    #"Weather underground data/Weather Underground East Boston Full 2023.csv",
    #"Weather underground data/Weather Underground East Boston Full 2024.csv",
    "Weather underground data/Weather Underground East Boston Jan-Nov 2025.csv",
]

df_list = []

for file in files:
    df_raw = pd.read_csv(file, low_memory=False)
    print(f"Loaded {file} with shape {df_raw.shape}")
    
    # normalize fog column names just in case (even if we dont use them now)
    rename_dict = {}
    for col in df_raw.columns:
        col_norm = col.replace(" ", "").lower()
        if col_norm == "dew/tempfog?":
            rename_dict[col] = "Dew/Temp fog?"
        if col_norm == "rhfog?":
            rename_dict[col] = "RH fog?"
    if rename_dict:
        df_raw = df_raw.rename(columns=rename_dict)
    
    df_list.append(df_raw)

# single raw dataframe with all years
df_raw = pd.concat(df_list, ignore_index=True)

display(df_raw.head(3))
df_raw.shape


Loaded Weather underground data/Weather Underground East Boston Jan-Nov 2025.csv with shape (86508, 15)


,Date,Time,Temperature_C,Dew_Point_C,Humidity_%,Wind,Speed_kmh,Gust_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,UV,Solar_w/m2,Dew/Temp fog?,RH fog?
0,1/1/2025,12:04 AM,6.39,5.11,91.0,East,5.95,13.84,1005.76,0.0,0.0,0,0.0,1,0
1,1/1/2025,12:09 AM,6.50,5.22,91.0,East,9.01,18.02,1006.10,0.0,0.0,0,0.0,1,0
2,1/1/2025,12:14 AM,6.61,5.28,91.0,East,7.40,16.09,1005.76,0.0,0.0,0,0.0,1,0


(86508, 15)

In [27]:
# combine Date + Time into local Boston time
dt_str = df_raw["Date"].astype(str) + " " + df_raw["Time"].astype(str)
ts_local = pd.to_datetime(dt_str, errors="coerce")

# localize to America/New_York and convert to UTC
ts_utc = (
    ts_local
    .dt.tz_localize("America/New_York", ambiguous="NaT", nonexistent="NaT")
    .dt.tz_convert("UTC")
)

df_raw["timestamp_utc"] = ts_utc
df_raw = df_raw.dropna(subset=["timestamp_utc"]).copy()

# Choose only column required
df = df_raw[[
    "timestamp_utc",
    "Temperature_C",
    "Dew_Point_C",
    "Humidity_%",
    "Wind",
    "Speed_kmh",
    "Pressure_hPa",
    "Precip_Rate_mm",
    "Precip_Accum_mm",
    "Dew/Temp fog?",
    "RH fog?",
]].copy()

df.info()


C:\Users\USER\AppData\Local\Temp\ipykernel_102884\1315111008.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts_local = pd.to_datetime(dt_str, errors="coerce")


<class 'pandas.core.frame.DataFrame'>
Index: 86484 entries, 0 to 86507
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp_utc    86484 non-null  datetime64[ns, UTC]
 1   Temperature_C    86462 non-null  float64            
 2   Dew_Point_C      86462 non-null  float64            
 3   Humidity_%       86462 non-null  float64            
 4   Wind             85174 non-null  object             
 5   Speed_kmh        86484 non-null  float64            
 6   Pressure_hPa     86484 non-null  float64            
 7   Precip_Rate_mm   86484 non-null  float64            
 8   Precip_Accum_mm  86484 non-null  float64            
 9   Dew/Temp fog?    86484 non-null  object             
 10  RH fog?          86484 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(7), int64(1), object(2)
memory usage: 7.9+ MB


In [28]:
# ensure numeric typing for some required variables
columns = [
    "Temperature_C",
    "Dew_Point_C",
    "Speed_kmh",
    "Pressure_hPa",
    "Precip_Rate_mm",
    "Precip_Accum_mm",
    "Dew/Temp fog?",
    "RH fog?",
]

for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86484 entries, 0 to 86507
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp_utc    86484 non-null  datetime64[ns, UTC]
 1   Temperature_C    86462 non-null  float64            
 2   Dew_Point_C      86462 non-null  float64            
 3   Humidity_%       86462 non-null  float64            
 4   Wind             85174 non-null  object             
 5   Speed_kmh        86484 non-null  float64            
 6   Pressure_hPa     86484 non-null  float64            
 7   Precip_Rate_mm   86484 non-null  float64            
 8   Precip_Accum_mm  86484 non-null  float64            
 9   Dew/Temp fog?    86462 non-null  float64            
 10  RH fog?          86484 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(8), int64(1), object(1)
memory usage: 7.9+ MB


### Resampling
This dataset is not at hourly resolution, so we need to resample it to hourly resolution.

In [29]:
# set index for resampling
df = df.set_index("timestamp_utc").sort_index()

# aggregate to hourly resolution
hourly = (
    df
    .resample("5T")
    .agg({
        "Temperature_C": "mean",
        "Dew_Point_C": "mean",
        "Speed_kmh": "mean",
        "Pressure_hPa": "mean",
        "Precip_Rate_mm": "mean", 
        "Precip_Accum_mm": "max", # this is cumulative; hourly max is a dumb down
        "Wind": "last", # wind direction is categorical; we take the last observation in the hour for now, change later if needed
        "Dew/Temp fog?": "max",
        "RH fog?": "max",
    })
    .reset_index()
)
print(hourly.shape)
hourly.head()

(91296, 10)


C:\Users\USER\AppData\Local\Temp\ipykernel_102884\1676918121.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  .resample("5T")


,timestamp_utc,Temperature_C,Dew_Point_C,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,Dew/Temp fog?,RH fog?
0,2025-01-01 05:00:00+00:00,6.39,5.11,5.95,1005.76,0.0,0.0,East,1.0,0.0
1,2025-01-01 05:05:00+00:00,6.50,5.22,9.01,1006.10,0.0,0.0,East,1.0,0.0
2,2025-01-01 05:10:00+00:00,6.61,5.28,7.40,1005.76,0.0,0.0,East,1.0,0.0
3,2025-01-01 05:15:00+00:00,6.72,5.39,7.88,1005.76,0.0,0.0,East,1.0,0.0
4,2025-01-01 05:20:00+00:00,6.78,5.50,7.24,1005.42,0.0,0.0,ENE,1.0,0.0


In [30]:
hourly.to_csv("Weather_Underground_5min_East_Boston_full.csv", index=False)